<a href="https://colab.research.google.com/github/muzzamilmehmood28-ops/Machine-learning-Internship-/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Load the dataset
df = pd.read_csv('Dataset.csv')


display(df.head(3))

# Check shape and columns
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"


Dataset Shape: (8790, 10)
Columns: ['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']


In [5]:
# Handle any potential missing values by replacing them with empty strings
df = df.fillna('')

# Create a new combined feature column for text processing
df['combined_features'] = df['listed_in'] + ' ' + df['director'] + ' ' + df['country'] + ' ' + df['rating']

# View a sample of the combined features
print(df['combined_features'].head(2).values)

['Documentaries Kirsten Johnson United States PG-13'
 'Crime TV Shows, International TV Shows, TV Action & Adventure Julien Leclercq France TV-MA']


In [6]:
# Initialize the TF-IDF Vectorizer (removing standard English stop words like 'the', 'and', etc.)
tfidf = TfidfVectorizer(stop_words='english')

# Construct the required TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

# Check the shape of the TF-IDF matrix
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (8790, 6568)


In [7]:
# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

Cosine Similarity Matrix Shape: (8790, 8790)


In [10]:
def recommend_titles(title, cosine_sim=cosine_sim, df=df):
    # Match title case-insensitively
    matches = df[df['title'].str.lower() == title.lower()]

    if matches.empty:
        return f"Title '{title}' not found in the dataset."

    # Get the index of the title that matches the input
    idx = matches.index[0]

    # Get the pairwise similarity scores of all titles with that title
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the titles based on the similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top 5 most similar titles (excluding the movie itself at index 0)
    sim_scores = sim_scores[1:6]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 5 most similar titles
    return df['title'].iloc[movie_indices].tolist()

In [15]:
# Test the recommendation engine
input_title = "Ganglands"
recommendations = recommend_titles(input_title)

print(f"Because you watched '{input_title}', you might like:")
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

Because you watched 'Ganglands', you might like:
1. Sentinelle
2. Earth and Blood
3. Lupin
4. Crime Time
5. Mortel
